# Experiment 15 — Phoneme Centroid Mapping

**What's missing from the universal language pipeline**: we now have semantic geometry
(Exp 4–9), a draft vocabulary (Exp 11), and the beginning of a grammar (Exp 9 / Exp 10).
What we don't have is **surface form** — the actual pronunciation of each proto-token.
This is not cosmetic: the phoneme inventory constrains vocabulary size, learnability, and
which existing languages the universal language will feel closest to for different speaker
populations.

**This experiment**:
1. For each concept in the universal vocabulary (loaded from Exp 11 CSV or from a
   built-in seed set), find the nearest word in each language
2. Convert each word to its phoneme sequence using `epitran` (G2P for 10+ languages)
3. Compute a **phoneme consensus**: which phoneme patterns appear across the most languages
   weighted by speaker population
4. Check the resulting proto-token phoneme inventory against WALS universal phonological
   constraints — does the data-driven inventory contain the features all languages share?
5. Produce a **draft pronunciation** for each proto-token and estimate how close it sounds
   to existing words in each language (phoneme edit distance)

**Key question**: does speaker-count-weighted phoneme consensus naturally produce an
inventory that satisfies Chomsky/WALS universal phonological constraints, or must
those be enforced explicitly?

In [ ]:
# pip install skipped - no internet in this environment
# eng_to_ipa is handled by the offline shim below
# sentence-transformers and scikit-learn assumed pre-installed

In [ ]:
# ── Offline eng_to_ipa replacement (no internet available) ───────────────────
import re, sys

_CMU_FALLBACK = {
    'water': 'wɔtər', 'fire': 'faɪər', 'earth': 'ɜrθ', 'wind': 'wɪnd',
    'sun': 'sʌn', 'moon': 'mun', 'hand': 'hænd', 'eye': 'aɪ',
    'mouth': 'maʊθ', 'head': 'hɛd', 'heart': 'hɑrt', 'blood': 'blʌd',
    'stone': 'stoʊn', 'tree': 'tri', 'fish': 'fɪʃ', 'bird': 'bɜrd',
    'dog': 'dɔg', 'child': 'tʃaɪld', 'mother': 'mʌðər', 'father': 'fɑðər',
    'eat': 'it', 'drink': 'drɪŋk', 'sleep': 'slip', 'walk': 'wɔk',
    'run': 'rʌn', 'see': 'si', 'hear': 'hɪr', 'speak': 'spik',
    'one': 'wʌn', 'two': 'tu', 'three': 'θri', 'big': 'bɪg',
    'small': 'smɔl', 'good': 'gʊd', 'bad': 'bæd', 'new': 'nu',
    'old': 'oʊld', 'long': 'lɔŋ', 'short': 'ʃɔrt', 'name': 'neɪm',
    'path': 'pæθ', 'night': 'naɪt', 'day': 'deɪ', 'year': 'jɪr',
    'person': 'pɜrsən', 'woman': 'wʊmən', 'man': 'mæn', 'come': 'kʌm',
    'go': 'goʊ', 'give': 'gɪv', 'take': 'teɪk', 'know': 'noʊ',
    'think': 'θɪŋk', 'want': 'wɑnt', 'say': 'seɪ', 'make': 'meɪk',
    'life': 'laɪf', 'death': 'dɛθ', 'love': 'lʌv', 'fear': 'fɪr',
    'light': 'laɪt', 'dark': 'dɑrk', 'time': 'taɪm', 'feel': 'fil',
    'sky': 'skaɪ', 'peace': 'pis', 'war': 'wɔr', 'home': 'hoʊm',
    'food': 'fud', 'pain': 'peɪn', 'joy': 'dʒɔɪ', 'hope': 'hoʊp',
}

_RULES = [
    (r'tion', 'ʃən'), (r'sion', 'ʒən'), (r'ght', 't'), (r'ph', 'f'),
    (r'ch', 'tʃ'), (r'sh', 'ʃ'), (r'th', 'θ'), (r'ng', 'ŋ'),
    (r'ee|ea', 'i'), (r'oo', 'u'), (r'ou|ow', 'aʊ'), (r'oi|oy', 'ɔɪ'),
    (r'ai|ay', 'eɪ'), (r'oa', 'oʊ'), (r'ie|igh', 'aɪ'),
    (r'e$', ''), (r'c(?=[ei])', 's'), (r'c', 'k'), (r'x', 'ks'),
    (r'q', 'k'), (r'y(?=[aeiou])', 'j'), (r'y$', 'i'),
]

def _rule_g2p(word):
    w = word.lower()
    for pat, repl in _RULES:
        w = re.sub(pat, repl, w)
    return w

class _IpaShim:
    """Drop-in replacement for eng_to_ipa (offline mode)."""
    def convert(self, word):
        return _CMU_FALLBACK.get(word.lower().strip(), _rule_g2p(word))
    def isin_cmu(self, word):
        return word.lower().strip() in _CMU_FALLBACK

sys.modules['eng_to_ipa'] = _IpaShim()
print('eng_to_ipa shim loaded (offline mode)')


In [ ]:
# Safe pip install - won't crash if network is unavailable
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "sentence-transformers", "scikit-learn"],
    capture_output=True, text=True
)
print("pip exit code:", result.returncode, "(network errors ignored in offline env)")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import re
from collections import Counter, defaultdict
from itertools import combinations

import eng_to_ipa as ipa
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer('LaBSE')
print('LaBSE and eng_to_ipa loaded')

# ── Phoneme classification using IPA feature heuristics ──────────────────────
# We classify IPA characters by their place/manner - a lightweight substitute
# for panphon that works without a C extension or separate data package.

STOPS       = set('pbtdkgʔɡ')
NASALS      = set('mnŋɲɳɴ')
FRICATIVES  = set('fvszʃʒθðxɣhħɸβ')
APPROXIMANTS= set('wjlrɹɾɻɭʎ')
VOWELS      = set('aeiouæɑɒɔɛɜɪʊʌəɐɨɵɯøœ')
AFFRICATES  = {'tʃ','dʒ','ts','dz','tɕ','dʑ'}

def classify_phoneme(ph):
    if ph in AFFRICATES:
        return 'stop'   # treated as stops for inventory purposes
    if ph and ph[0] in STOPS:
        return 'stop'
    if ph and ph[0] in NASALS:
        return 'nasal'
    if ph and ph[0] in FRICATIVES:
        return 'fricative'
    if ph and ph[0] in VOWELS:
        return 'vowel'
    if ph and ph[0] in APPROXIMANTS:
        return 'approximant'
    return 'other'

print('Phoneme classifier ready')

In [ ]:
# ── Language configuration ──────────────────────────────────────────────
LANGS = ['en', 'es', 'fr', 'de', 'pt', 'it']

SPEAKER_WEIGHTS = {
    'en': 1500, 'es': 560, 'fr': 280,
    'de': 130,  'pt': 260, 'it': 85,
}

# Lightweight rule-based G2P tables per language.
# Each rule: (regex_pattern_string, replacement_string)
# Applied sequentially to lowercase word. Sufficient for phoneme frequency analysis.
import re

G2P_RULES = {
    'es': [
        ('ch', 'tS'), ('ll', 'L'), ('ny', 'N'), ('qu', 'k'),
        ('gu', 'g'), ('c(?=[ei])', 's'), ('c', 'k'),
        ('g(?=[ei])', 'x'), ('j', 'x'), ('h', ''),
        ('rr', 'R'), ('v', 'b'), ('z', 's'),
        ('[aeiou]', lambda m: m.group()),
    ],
    'fr': [
        ('ch', 'S'), ('ph', 'f'), ('gn', 'N'), ('qu', 'k'),
        ('eau', 'o'), ('au', 'o'), ('eu', 'e'), ('ou', 'u'),
        ('ai|ei', 'e'), ('oi', 'wa'),
        ('j', 'Z'), ('g(?=[ei])', 'Z'), ('c(?=[ei])', 's'), ('c', 'k'),
        ('r', 'R'), ('[eéèê]', 'e'), ('[àâ]', 'a'), ('h', ''),
    ],
    'de': [
        ('sch', 'S'), ('ch(?=[ei])', 'C'), ('ch', 'x'), ('ck', 'k'),
        ('ph', 'f'), ('tz', 'ts'), ('z', 'ts'), ('v', 'f'), ('w', 'v'),
        ('[uU]', 'u'), ('[oO]', 'o'), ('ae', 'e'), ('oe', 'e'), ('ue', 'y'),
        ('ei|ai', 'aI'), ('au', 'aU'), ('eu|au', 'OY'),
        ('ng', 'N'), ('ss|sz', 's'),
    ],
    'pt': [
        ('ch', 'S'), ('lh', 'L'), ('nh', 'N'), ('qu', 'k'),
        ('gu(?=[ei])', 'g'), ('c(?=[ei])', 's'), ('c', 'k'),
        ('g(?=[ei])', 'Z'), ('j', 'Z'), ('x', 'S'),
        ('rr', 'R'), ('v', 'v'), ('h', ''),
    ],
    'it': [
        ('ch', 'k'), ('gh', 'g'), ('sc(?=[ei])', 'S'), ('gn', 'N'),
        ('c(?=[ei])', 'tS'), ('c', 'k'),
        ('g(?=[ei])', 'dZ'), ('g', 'g'),
        ('z', 'ts'), ('h', ''),
    ],
}

def apply_g2p(word, lang):
    """Apply rule-based G2P. Returns list of character-level phoneme strings."""
    import eng_to_ipa as ipa_mod
    if lang == 'en':
        result = ipa_mod.convert(word.lower()).replace('*', '')
        return [c for c in result if c.strip() and c not in 'u02c8u02ccu002e']
    rules = G2P_RULES.get(lang, [])
    s = word.lower()
    for pattern, repl in rules:
        try:
            s = re.sub(pattern, repl, s)
        except Exception:
            pass
    return [c for c in s if c.strip()]

# Sanity check
test_words = [('water','en'), ('agua','es'), ('eau','fr'), ('Wasser','de')]
for w, l in test_words:
    phones = apply_g2p(w, l)
    print(f'  {l}: {w} -> {phones}')
print('G2P rules ready')


In [ ]:
# ── Concept vocabulary ────────────────────────────────────────────────────────
# Try to load from Exp 11 output. If not present, use the built-in seed set.

import os

if os.path.exists('/kaggle/working/exp11_universal_vocab.csv'):
    vocab_df = pd.read_csv('/kaggle/working/exp11_universal_vocab.csv')
    supported = list(EPITRAN_CODES.keys()) if 'EPITRAN_CODES' in dir() else LANGS
    for lang in supported:
        if lang not in vocab_df.columns:
            vocab_df[lang] = np.nan
    mask = vocab_df[supported].notna().all(axis=1)
    vocab_df = vocab_df[mask].head(100).reset_index(drop=True)
    print(f'Loaded {len(vocab_df)} concepts from Exp 11 output')
else:
    SEED_VOCAB = [
        {'en':'water', 'es':'agua',      'fr':'eau',       'de':'Wasser',   'pt':'água',    'it':'acqua'},
        {'en':'fire',  'es':'fuego',     'fr':'feu',       'de':'Feuer',    'pt':'fogo',    'it':'fuoco'},
        {'en':'earth', 'es':'tierra',    'fr':'terre',     'de':'Erde',     'pt':'terra',   'it':'terra'},
        {'en':'sky',   'es':'cielo',     'fr':'ciel',      'de':'Himmel',   'pt':'céu',     'it':'cielo'},
        {'en':'love',  'es':'amor',      'fr':'amour',     'de':'Liebe',    'pt':'amor',    'it':'amore'},
        {'en':'fear',  'es':'miedo',     'fr':'peur',      'de':'Angst',    'pt':'medo',    'it':'paura'},
        {'en':'trust', 'es':'confianza', 'fr':'confiance', 'de':'Vertrauen','pt':'confiança','it':'fiducia'},
        {'en':'light', 'es':'luz',       'fr':'lumière',   'de':'Licht',    'pt':'luz',     'it':'luce'},
        {'en':'dark',  'es':'oscuridad', 'fr':'obscurité', 'de':'Dunkel',   'pt':'escuridão','it':'oscurità'},
        {'en':'time',  'es':'tiempo',    'fr':'temps',     'de':'Zeit',     'pt':'tempo',   'it':'tempo'},
        {'en':'life',  'es':'vida',      'fr':'vie',       'de':'Leben',    'pt':'vida',    'it':'vita'},
        {'en':'death', 'es':'muerte',    'fr':'mort',      'de':'Tod',      'pt':'morte',   'it':'morte'},
        {'en':'eat',   'es':'comer',     'fr':'manger',    'de':'essen',    'pt':'comer',   'it':'mangiare'},
        {'en':'sleep', 'es':'dormir',    'fr':'dormir',    'de':'schlafen', 'pt':'dormir',  'it':'dormire'},
        {'en':'run',   'es':'correr',    'fr':'courir',    'de':'laufen',   'pt':'correr',  'it':'correre'},
        {'en':'give',  'es':'dar',       'fr':'donner',    'de':'geben',    'pt':'dar',     'it':'dare'},
        {'en':'take',  'es':'tomar',     'fr':'prendre',   'de':'nehmen',   'pt':'tomar',   'it':'prendere'},
        {'en':'speak', 'es':'hablar',    'fr':'parler',    'de':'sprechen', 'pt':'falar',   'it':'parlare'},
        {'en':'think', 'es':'pensar',    'fr':'penser',    'de':'denken',   'pt':'pensar',  'it':'pensare'},
        {'en':'feel',  'es':'sentir',    'fr':'ressentir', 'de':'fühlen',   'pt':'sentir',  'it':'sentire'},
        {'en':'wind',  'es':'viento',    'fr':'vent',      'de':'Wind',     'pt':'vento',   'it':'vento'},
        {'en':'stone', 'es':'piedra',    'fr':'pierre',    'de':'Stein',    'pt':'pedra',   'it':'pietra'},
        {'en':'river', 'es':'río',       'fr':'rivière',   'de':'Fluss',    'pt':'rio',     'it':'fiume'},
        {'en':'child', 'es':'niño',      'fr':'enfant',    'de':'Kind',     'pt':'criança', 'it':'bambino'},
        {'en':'friend','es':'amigo',     'fr':'ami',       'de':'Freund',   'pt':'amigo',   'it':'amico'},
        {'en':'peace', 'es':'paz',       'fr':'paix',      'de':'Frieden',  'pt':'paz',     'it':'pace'},
        {'en':'war',   'es':'guerra',    'fr':'guerre',    'de':'Krieg',    'pt':'guerra',  'it':'guerra'},
        {'en':'pain',  'es':'dolor',     'fr':'douleur',   'de':'Schmerz',  'pt':'dor',     'it':'dolore'},
        {'en':'joy',   'es':'alegría',   'fr':'joie',      'de':'Freude',   'pt':'alegria', 'it':'gioia'},
        {'en':'hope',  'es':'esperanza', 'fr':'espoir',    'de':'Hoffnung', 'pt':'esperança','it':'speranza'},
        {'en':'dream', 'es':'sueño',     'fr':'rêve',      'de':'Traum',    'pt':'sonho',   'it':'sogno'},
        {'en':'build', 'es':'construir', 'fr':'construire','de':'bauen',    'pt':'construir','it':'costruire'},
        {'en':'break', 'es':'romper',    'fr':'casser',    'de':'brechen',  'pt':'quebrar', 'it':'rompere'},
        {'en':'find',  'es':'encontrar', 'fr':'trouver',   'de':'finden',   'pt':'encontrar','it':'trovare'},
        {'en':'lose',  'es':'perder',    'fr':'perdre',    'de':'verlieren','pt':'perder',  'it':'perdere'},
        {'en':'change','es':'cambiar',   'fr':'changer',   'de':'ändern',   'pt':'mudar',   'it':'cambiare'},
        {'en':'new',   'es':'nuevo',     'fr':'nouveau',   'de':'neu',      'pt':'novo',    'it':'nuovo'},
        {'en':'old',   'es':'viejo',     'fr':'vieux',     'de':'alt',      'pt':'velho',   'it':'vecchio'},
        {'en':'hand',  'es':'mano',      'fr':'main',      'de':'Hand',     'pt':'mão',     'it':'mano'},
        {'en':'eye',   'es':'ojo',       'fr':'oeil',      'de':'Auge',     'pt':'olho',    'it':'occhio'},
    ]
    vocab_df = pd.DataFrame(SEED_VOCAB)
    print(f'Using built-in seed vocabulary: {len(vocab_df)} concepts')

print(vocab_df.head(3).to_string())

In [ ]:
# ── Step 1: G2P - convert each word to IPA phoneme sequence ──────────────────
print('Running G2P on vocabulary...')
phoneme_data = []

for _, row in vocab_df.iterrows():
    concept = row['en']
    for lang in LANGS:
        if lang not in row.index or pd.isna(row.get(lang)):
            continue
        word     = str(row[lang])
        phonemes = apply_g2p(word, lang)
        phoneme_data.append({
            'concept':  concept,
            'lang':     lang,
            'word':     word,
            'phonemes': phonemes,
            'n_phones': len(phonemes),
        })

phoneme_df = pd.DataFrame(phoneme_data)
print(f'G2P complete: {len(phoneme_df)} (concept, language) pairs')
print(f'Average phones per word: {phoneme_df["n_phones"].mean():.1f}')
print(f'Empty results: {(phoneme_df["n_phones"]==0).sum()}')
print()
print(phoneme_df[phoneme_df['n_phones']>0].head(10)[['concept','lang','word','phonemes']].to_string(index=False))

In [ ]:
# ── Step 2: Phoneme consensus algorithm ──────────────────────────────────────
# For each concept:
#   1. Collect all phoneme sequences across languages, each weighted by speaker count
#   2. Compute a phoneme frequency distribution (weighted) across all positions
#   3. Select the proto-token as the shortest sequence that preserves the most
#      speaker-weighted phoneme mass
#
# Position-free approach: we count unigram phoneme frequencies globally
# (positional alignment across languages with different syllable structures is
# ambiguous - we defer that to a future experiment)

def phoneme_consensus(concept, phoneme_df, speaker_weights):
    """
    Returns:
      weighted_phoneme_freq: Counter of phoneme → weighted count
      proto_token: 2–4 phoneme proto-token string
      coverage: fraction of speaker-weight covered by proto_token phonemes
    """
    sub = phoneme_df[phoneme_df['concept'] == concept]
    if len(sub) == 0:
        return None

    total_weight = 0.0
    freq = Counter()

    for _, row in sub.iterrows():
        w = speaker_weights.get(row['lang'], 50)
        for ph in row['phonemes']:
            freq[ph] += w
        total_weight += w

    # Normalise
    for ph in freq:
        freq[ph] /= (total_weight + 1e-9)

    # Proto-token: top phonemes by frequency, capped at 4
    top_phones  = [ph for ph, _ in freq.most_common(4)]
    proto_token = ''.join(top_phones)

    # Coverage: fraction of total phoneme mass covered by proto_token phonemes
    coverage = sum(freq[ph] for ph in top_phones)

    return {
        'concept':    concept,
        'freq':       freq,
        'proto_token': proto_token,
        'coverage':   round(coverage, 4),
        'n_langs':    len(sub),
    }

consensus_results = []
for concept in vocab_df['en'].unique():
    r = phoneme_consensus(concept, phoneme_df, SPEAKER_WEIGHTS)
    if r:
        consensus_results.append(r)

consensus_df = pd.DataFrame([
    {'concept': r['concept'], 'proto_token': r['proto_token'],
     'coverage': r['coverage'], 'n_langs': r['n_langs']}
    for r in consensus_results
])
consensus_df = consensus_df.sort_values('coverage', ascending=False)

print('Proto-token assignments (sorted by phoneme coverage):')
print(consensus_df.to_string(index=False))

In [ ]:
# ── Step 3: WALS constraint check ─────────────────────────────────────────────
# WALS near-universals we check:
#   1. All languages have stops (plosives)
#   2. All languages have nasals
#   3. All languages have at least one fricative
#   4. All languages have vowels
# We check whether these appear in the aggregate proto-token inventory.

# Build global inventory from all concept phoneme frequency distributions
global_inventory = Counter()
for r in consensus_results:
    for ph, freq in r['freq'].items():
        global_inventory[ph] += freq

if not global_inventory:
    print("WARNING: global_inventory is empty - G2P produced no phonemes.")
    print("Check that vocab_df has populated language columns and G2P rules fire correctly.")
else:
    inventory_df = pd.DataFrame([
        {'phoneme': ph, 'weight': w, 'class': classify_phoneme(ph)}
        for ph, w in global_inventory.most_common(50)
    ])

    print(f'Global phoneme inventory: {len(inventory_df)} distinct segments')
    print()
    print(inventory_df.head(30).to_string(index=False))

    classes_present = set(inventory_df['class'].unique())
    print()
    print('━━━ WALS Near-Universal Constraint Checks ━━━')
    checks = [
        ('stops (plosives)',    'stop',        'stop'        in classes_present),
        ('nasals',             'nasal',       'nasal'       in classes_present),
        ('fricatives',         'fricative',   'fricative'   in classes_present),
        ('vowels (syllabic)',  'vowel',       'vowel'       in classes_present),
        ('approximants',       'approximant', 'approximant' in classes_present),
    ]
    for name, cls, present in checks:
        status = '✓  PRESENT' if present else '✗  MISSING - explicit design needed'
        print(f'  {name:<25} {status}')

In [ ]:
# ── Step 4: Phoneme edit distance - how close does each proto-token sound? ───
# For each concept, compute the phoneme edit distance between the proto-token
# and the nearest word in each language.
# Low edit distance = speakers of that language will find the proto-token easy to pronounce.

def phoneme_edit_distance(seq1, seq2):
    """
    Levenshtein distance on lists of phoneme strings.
    Simple implementation - for production use 'phonological distance' metrics.
    """
    m, n = len(seq1), len(seq2)
    dp = np.zeros((m+1, n+1), dtype=int)
    for i in range(m+1):
        dp[i][0] = i
    for j in range(n+1):
        dp[0][j] = j
    for i in range(1, m+1):
        for j in range(1, n+1):
            cost = 0 if seq1[i-1] == seq2[j-1] else 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)
    return int(dp[m][n])

# Compute edit distances for each concept × language
print('Computing phoneme edit distances...')
edit_records = []

for r in consensus_results:
    concept   = r['concept']
    # proto-token as phoneme list (re-derive from top-4 phonemes)
    proto_phones = list(r['freq'].keys())[:4]

    sub = phoneme_df[phoneme_df['concept'] == concept]
    for _, row in sub.iterrows():
        if not row['phonemes']:
            continue
        ed = phoneme_edit_distance(proto_phones, row['phonemes'])
        # normalise by max sequence length
        max_len = max(len(proto_phones), len(row['phonemes']))
        ed_norm = ed / max(max_len, 1)
        edit_records.append({
            'concept':   concept,
            'lang':      row['lang'],
            'word':      row['word'],
            'proto':     ''.join(proto_phones),
            'ed':        ed,
            'ed_norm':   round(ed_norm, 3),
        })

edit_df = pd.DataFrame(edit_records)
print(f'Edit distances computed: {len(edit_df)} pairs')
print(f'Mean normalised edit distance: {edit_df["ed_norm"].mean():.3f}')
print(f'  (0 = proto-token is identical to existing word; 1 = maximally different)')

## Visualisation

In [ ]:
# ── Figure 1: Phoneme inventory ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel A: Top 20 phonemes by weighted frequency, coloured by class
ax = axes[0]
top20 = inventory_df.head(20)
class_colors = {
    'stop': '#E24B4A', 'nasal': '#1D9E75', 'fricative': '#378ADD',
    'vowel': '#EF9F27', 'approximant': '#7F77DD', 'other': '#AAAAAA'
}
colors = [class_colors.get(c, '#AAAAAA') for c in top20['class']]
bars = ax.barh(top20['phoneme'], top20['weight'], color=colors, height=0.7)
ax.set_xlabel('Speaker-weighted frequency in proto-tokens')
ax.set_title('Universal proto-token phoneme inventory\n(coloured by phoneme class)')
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=v, label=k) for k, v in class_colors.items() if k != 'other']
ax.legend(handles=legend_els, fontsize=9)
ax.grid(True, alpha=0.2, axis='x')

# Panel B: Phoneme class breakdown (pie chart)
ax = axes[1]
class_weights = inventory_df.groupby('class')['weight'].sum()
class_weights = class_weights[class_weights > 0]
wedge_colors = [class_colors.get(c, '#AAAAAA') for c in class_weights.index]
ax.pie(class_weights, labels=class_weights.index, colors=wedge_colors,
       autopct='%1.1f%%', startangle=90, pctdistance=0.8)
ax.set_title('Phoneme class distribution in\nproto-token inventory')

plt.suptitle('Experiment 15 - Global Phoneme Inventory of Proto-Tokens', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp15_phoneme_inventory.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: Edit distance heatmap (concept × language) ─────────────────────
fig, ax = plt.subplots(figsize=(12, max(6, len(vocab_df)*0.35)))

pivot = edit_df.pivot_table(index='concept', columns='lang', values='ed_norm', aggfunc='mean')
# Sort concepts by mean edit distance across all languages
pivot['mean'] = pivot.mean(axis=1)
pivot = pivot.sort_values('mean').drop(columns='mean')

mask = pivot.isna()
sns.heatmap(pivot, ax=ax, cmap='RdYlGn_r', annot=True, fmt='.2f',
            mask=mask, linewidths=0.3,
            cbar_kws={'label': 'Normalised phoneme edit distance\n(0=identical, 1=max different)'},
            vmin=0, vmax=1)
ax.set_title('Phoneme edit distance: proto-token vs nearest word per language\n'
             '(green = proto-token sounds like the existing word; '
             'red = needs significant phonological adjustment)')
ax.set_xlabel('Language')
ax.set_ylabel('Concept')
plt.tight_layout()
plt.savefig('exp15_edit_distance_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretation:')
print('  Green rows = concept proto-token sounds natural across many languages.')
print('  Red rows = proto-token phonology is foreign-feeling - may need explicit design.')
print('  Dark red cells = a specific language-concept pair needs a bridge pronunciation.')

In [ ]:
# ── Figure 3: Proto-token draft vocabulary ────────────────────────────────────
# Show the draft pronunciation next to the source words, sorted by coverage.

print('\nDraft universal pronunciation table:')
print(f'{"Concept":<12} {"Proto-token":<14} {"Coverage":>9}  '
      + '  '.join(f'{l:>6}' for l in LANGS))
print('-' * (12 + 14 + 12 + 8 * len(LANGS)))

for _, row in consensus_df.head(30).iterrows():
    concept = row['concept']
    proto   = row['proto_token']
    cov     = row['coverage']
    vocab_row = vocab_df[vocab_df['en'] == concept]
    if len(vocab_row) == 0:
        continue
    vocab_row = vocab_row.iloc[0]
    words = '  '.join(f"{str(vocab_row.get(l, '–'))[:6]:>6}" for l in LANGS)
    print(f'{concept:<12} /{proto}/ {cov:>12.3f}  {words}')

print()
print('Coverage = fraction of speaker-weighted phoneme mass captured by the 4-phoneme proto-token.')
print('Higher coverage → proto-token will be phonetically familiar to more speakers.')

In [ ]:
# ── Figure 4: Per-language phonetic proximity ─────────────────────────────────
# For each language, what is the average edit distance to all proto-tokens?
# This tells us which language population will find the universal language
# hardest vs easiest to pronounce.

per_lang_mean = edit_df.groupby('lang')['ed_norm'].agg(['mean','std']).reset_index()
per_lang_mean = per_lang_mean.sort_values('mean')

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#1D9E75','#378ADD','#EF9F27','#D4537E','#7F77DD','#E24B4A']
bars = ax.bar(per_lang_mean['lang'], per_lang_mean['mean'],
              yerr=per_lang_mean['std'], color=colors[:len(per_lang_mean)],
              capsize=4, alpha=0.85, width=0.6)
ax.set_ylabel('Mean normalised phoneme edit distance to proto-tokens')
ax.set_xlabel('Language')
ax.set_title('Phonetic proximity to universal proto-tokens per language\n'
             '(lower = proto-tokens sound more like this language)')
ax.grid(True, alpha=0.2, axis='y')
for bar, (_, row) in zip(bars, per_lang_mean.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+row['std']+0.01,
            f"{row['mean']:.3f}", ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('exp15_language_proximity.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nPer-language phonetic proximity ranking:')
for _, row in per_lang_mean.iterrows():
    bar_str = '█' * int((1 - row['mean']) * 30)
    print(f"  {row['lang']:4s} | {bar_str:<30} | mean edit = {row['mean']:.3f}")

In [ ]:
# ── Export draft pronunciation table ─────────────────────────────────────────
# consensus_df has 'concept'; vocab_df has 'en' - merge on those, then drop the
# redundant 'en' column only if it actually appears after the merge.

vocab_cols = ['en'] + [l for l in LANGS if l in vocab_df.columns]
vocab_sub  = vocab_df[vocab_cols].copy()
# Guard: drop any pre-existing 'concept' column before renaming 'en' → 'concept'
# (vocab_df sometimes carries a 'concept' column alongside 'en', causing a
#  duplicate label that pandas refuses to merge on)
if 'concept' in vocab_sub.columns:
    vocab_sub = vocab_sub.drop(columns=['concept'])
vocab_sub = vocab_sub.rename(columns={'en': 'concept'})

output = consensus_df.merge(vocab_sub, on='concept', how='left')
# Drop any accidentally duplicated columns
output = output.loc[:, ~output.columns.duplicated()]
output = output.sort_values('coverage', ascending=False)
output.to_csv('exp15_proto_token_pronunciations.csv', index=False)
print(f'Saved {len(output)} proto-token pronunciations to exp15_proto_token_pronunciations.csv')
print(output[['concept','proto_token','coverage'] + [l for l in LANGS if l in output.columns]].head(10).to_string(index=False))

# ── Summary ───────────────────────────────────────────────────────────────────
print()
print('━━━ Experiment 15 Summary ━━━')
print(f'  Concepts processed:     {len(consensus_df)}')
print(f'  Languages (G2P):        {len(LANGS)}')
if len(inventory_df) > 0:
    print(f'  Phoneme inventory size: {(inventory_df["class"] != "other").sum()} distinct segments')
print()
print('  WALS near-universal constraints:')
for name, cls, present in checks:
    print(f'    {name:<25} {"✓" if present else "✗"}')
print()
if len(edit_df) > 0:
    print(f'  Mean phoneme edit distance (proto → existing word): {edit_df["ed_norm"].mean():.3f}')
    print()
    print('  Most phonetically natural concepts (lowest mean edit distance):')
    for concept, mean_ed in edit_df.groupby('concept')['ed_norm'].mean().sort_values().head(5).items():
        print(f'    {concept:<12} {mean_ed:.3f}')
print()
print('  Next step → Experiment 10: feed proto-tokens into grammatical frame classifier')
print('  The phoneme inventory confirms/violates WALS universals - shape explicit grammar design')